# Laboratorium 6: Lokalne modele językowe

W tym laboratorium porównujemy lokalne modele językowe jako komponenty systemu NLP. Obok jakości różnych wariantów odpowiedzi modelu interesuje nas również koszt uruchomienia, pamięć, opóźnienie, szybkość generowania i niezawodność wyjścia strukturalnego.

Główne pytanie laboratorium:

**Jak wybrać lokalny LLM, kiedy jednocześnie liczą się jakość, pamięć, opóźnienie i przewidywalność działania?**

Porównujemy co najmniej dwa lokalne modele lub dwie wyraźnie różne konfiguracje modeli.


## Po tych zajęciach student potrafi

- opisać środowisko uruchomieniowe lokalnego LLM;
- zebrać podstawowe informacje o modelach i kwantyzacji;
- oszacować koszt pamięci wag;
- zmierzyć cold start, TTFT i szybkość generowania;
- przygotować powtarzalny benchmark jakości;
- ocenić wyjście strukturalne;
- porównać dwie kwantyzacje tego samego modelu;
- porównać LLM z klasycznym baseline'em;
- analizować błędy modeli generatywnych;
- uzasadnić wybór modelu dla konkretnego zastosowania.


## Protokół

Wszystkie modele porównujemy na tym samym sprzęcie, na tych samych danych i przy możliwie zgodnych parametrach generowania. Jeżeli z powodów sprzętowych nie można uruchomić wybranego modelu, należy opisać ograniczenie i wybrać mniejszą konfigurację.

Nie zmieniamy promptu osobno dla każdego modelu tylko po to, aby poprawić jego wynik. Prompt stanowi część wspólnego benchmarku.

Parametry generowania są wspólne i zamrożone w `GEN_OPTIONS`: `temperature=0`, stały `seed`, stałe `num_ctx`. Limit `num_predict` ustalamy osobno dla każdego zadania, ale identycznie dla wszystkich modeli. Dla modeli z trybem „thinking" wyłączamy go (`think=False`) albo raportujemy go jako osobną konfigurację.

Wszystkie surowe odpowiedzi zapisujemy do `artifacts/` (JSONL lub parquet: tu potrzebny `pyarrow`). Analiza w zadaniach 7-9, a także w zadaniu dla chętnych, korzysta z zapisanych plików, a nie z ponownego generowania.

In [ ]:

from pathlib import Path
import json, platform, time
import jsonschema
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, psutil

from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42
OLLAMA_BASE = "http://localhost:11434"
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 80)


In [ ]:
GEN_OPTIONS = {"temperature": 0, "seed": RANDOM_STATE, "num_ctx": 4096}

def ollama_get(path):
    r = requests.get(f"{OLLAMA_BASE}{path}", timeout=10)
    r.raise_for_status()
    return r.json()

# /api/version -> wersja, /api/tags -> modele na dysku, /api/ps -> modele w pamięci (size, size_vram)
print("Ollama:", ollama_get("/api/version")["version"])

def unload(model):
    """Usuwa model z pamięci; kolejne wywołanie będzie cold startem."""
    r = requests.post(f"{OLLAMA_BASE}/api/generate",
                      json={"model": model, "keep_alive": 0}, timeout=120)
    r.raise_for_status()

def chat(model, prompt, *, system=None, fmt=None, num_predict=128, think=None):
    """Jedno wywołanie /api/chat ze streamingiem. Zwraca tekst i metryki."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    payload = {"model": model, "messages": messages, "stream": True,
               "options": {**GEN_OPTIONS, "num_predict": num_predict}}
    if fmt is not None:
        payload["format"] = fmt          # "json" albo słownik ze schematem JSON
    if think is not None:
        payload["think"] = think

    t0, ttft, chunks, last = time.perf_counter(), None, [], {}
    with requests.post(f"{OLLAMA_BASE}/api/chat", json=payload,
                       stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line:
                continue
            part = json.loads(line)
            if part.get("error"):
                raise RuntimeError(f"Ollama /api/chat: {part['error']}")
            piece = (part.get("message") or {}).get("content") or ""
            if piece and ttft is None:
                ttft = time.perf_counter() - t0
            chunks.append(piece)
            if part.get("done"):
                last = part
    if not last:
        raise RuntimeError(f"Niekompletna odpowiedź /api/chat dla {model}.")
    eval_s = last.get("eval_duration", 0) / 1e9
    return {
        "model": model,
        "text": "".join(chunks),
        "ttft_s": ttft,                                   # zegar klienta
        "total_s": time.perf_counter() - t0,              # zegar klienta
        "load_s": last.get("load_duration", 0) / 1e9,     # raport serwera
        "prompt_tokens": last.get("prompt_eval_count"),
        "output_tokens": last.get("eval_count"),
        "tokens_per_s": last.get("eval_count", 0) / eval_s if eval_s else None,
        "done_reason": last.get("done_reason"),           # "length" = ucięte przez num_predict
    }

# Zadanie 1: Audyt środowiska i modeli

Proszę przygotować opis środowiska zawierający:

- system operacyjny;
- CPU;
- RAM;
- GPU i VRAM, jeśli są dostępne;
- wersję Ollamy;
- listę lokalnie dostępnych modeli.

Następnie proszę wybrać co najmniej dwa modele lub konfiguracje do dalszego porównania i zapisać dla nich nazwę, rodzinę, przybliżoną liczbę parametrów, kwantyzację oraz rozmiar pliku.

Wynik proszę zapisać również do `artifacts/local_llm_environment.json`.


Zalecana konfiguracja (proszę pobrać przed zajęciami):

| rola | tag | przybliżony rozmiar |
|---|---|---:|
| mały model | `gemma3:1b` | 0.8 GB |
| większy model | `gemma3:4b` | 3.3 GB |
| druga kwantyzacja (zadanie 6) | `gemma3:4b-it-q8_0` | ok. 5 GB |

Przy 8 GB RAM i braku GPU zadanie 6 można wykonać na parze `gemma3:1b` (Q4_K_M) i `gemma3:1b-it-q8_0`.

Opcjonalnie, jako model z innej rodziny: `qwen3:4b`. Ten model ma tryb „thinking", dlatego w benchmarku uruchamiamy go z `think=False`. Wariant z `think=True` można dodać jako osobną konfigurację i porównać TTFT. Można też użyć innego modelu z tej rodziny.

Inne modele są dopuszczalne, jeśli spełniają warunek zadania: co najmniej dwa rozmiary albo dwie kwantyzacje tego samego modelu.

In [ ]:
# Proszę zaimplementować audyt środowiska i inwentaryzację modeli.

# Zadanie 2: Pamięć i kwantyzacja

Proszę oszacować minimalny rozmiar samych wag dla modeli 1B, 4B i 8B przy zapisie FP16, INT8 i 4-bitowym.

Dla rzeczywiście uruchamianych modeli proszę porównać oszacowanie z rozmiarem pliku oraz, jeśli to możliwe, ze zmianą wykorzystania RAM/VRAM po załadowaniu modelu.

Proszę wyjaśnić, dlaczego rozmiar pliku, rozmiar samych wag i rzeczywiste zużycie pamięci nie muszą być identyczne.


Wskazówki:
- „4-bit" w plikach GGUF nie oznacza 4 bitów na każdą wagę. Proszę policzyć efektywną liczbę bitów na wagę jako `rozmiar_pliku * 8 / liczba_parametrów` i porównać ją z nominalną nazwą kwantyzacji.
- Ollama działa w osobnym procesie, więc pamięć procesu notebooka nic nie mówi o modelu. Proszę użyć `/api/ps` (pola `size` i `size_vram`) oraz różnicy `psutil.virtual_memory().used` przed załadowaniem i po nim.
- Proszę sprawdzić, jak zmienia się zużycie pamięci po zmianie `num_ctx` z 4096 na 8192 dla jednego modelu.

In [ ]:
# Proszę zaimplementować oszacowanie pamięci i zebrać pomiary dla wybranych modeli.

# Zadanie 3: Cold start, TTFT i szybkość generowania

Proszę użyć funkcji `chat()` i jednego krótkiego promptu z `num_predict=128`. Dla każdego modelu mierzymy dwa stany:

**Cold start** (3 powtórzenia, przed każdym `unload(model)`):
- `load_s` -- czas ładowania raportowany przez serwer;
- `ttft_s` -- czas do pierwszego fragmentu treści odpowiedzi, łącznie z ładowaniem.

**Stan ciepły** (jedno wywołanie rozgrzewające, potem 5 powtórzeń):
- `ttft_s`;
- `total_s`;
- `output_tokens`;
- `tokens_per_s`.

Proszę raportować medianę oraz min–max. Proszę skomentować, czy drugi i trzeci cold start różnią się od pierwszego i dlaczego (cache plików systemu operacyjnego).

In [ ]:
# Proszę zaimplementować wspólny benchmark wydajności lokalnych modeli.

# Zadanie 4: Klasyfikacja zero-shot

Proszę przygotować deterministyczną, możliwie zbalansowaną próbkę PolEmo 2.0 i jedną instrukcję klasyfikacyjną wspólną dla wszystkich modeli.

Dla każdego modelu proszę policzyć:

- accuracy;
- macro-F1;
- F1 dla każdej klasy;
- odsetek odpowiedzi, których nie da się poprawnie zinterpretować jako etykiety;
- medianę i średnią czasu na dokument (model rozgrzany);
- 95% przedział ufności macro-F1 (`macro_f1_ci`).

Odpowiedzi interpretujemy wyłącznie funkcją `parse_label`. Odpowiedź `INVALID` liczy się jako błąd klasyfikacji. Predykcje i surowe odpowiedzi proszę zapisać do `artifacts/lab6_classification_<model>.parquet`.

In [ ]:
DATA_PATH = ARTIFACTS / "polemo2_clean.parquet"
HF_BASE = "https://huggingface.co/datasets/allegro/klej-polemo2-in/resolve/main/"

if DATA_PATH.exists():
    data = pd.read_parquet(DATA_PATH)
else:
    for name in ("valid.csv", "dev.csv", "validation.csv"):
        try:
            data = pd.read_csv(HF_BASE + name)
            break
        except Exception:
            continue
    else:
        raise FileNotFoundError("Brak artefaktu z lab. 1 i nie udało się pobrać zbioru walidacyjnego.")
    data["split"] = "validation"

data = data.rename(columns={"sentence": "text"})

LABEL_MAP = {
    "__label__meta_minus_m": "negatywna",
    "__label__meta_plus_m": "pozytywna",
    "__label__meta_zero": "neutralna",
    "__label__meta_amb": "niejednoznaczna",
    "minus": "negatywna",
    "plus": "pozytywna",
    "zero": "neutralna",
    "amb": "niejednoznaczna",
    "negatywna": "negatywna",
    "pozytywna": "pozytywna",
    "neutralna": "neutralna",
    "niejednoznaczna": "niejednoznaczna",
}
LABELS = list(dict.fromkeys(LABEL_MAP.values()))
source_col = "label" if "label" in data.columns else "target"

data["label"] = (
    data[source_col]
    .astype(str)
    .str.strip()
    .map(LABEL_MAP)
)

if data["label"].isna().any():
    raise ValueError(
        "Nierozpoznane etykiety PolEmo. "
        "Sprawdź kodowanie i źródło danych."
    )

if "split" not in data.columns:
    raise ValueError(
        "Brak kolumny split. Wskaż jawnie zbiór walidacyjny."
    )

valid_df = data[
    data["split"].astype(str).str.lower().isin(
        ["validation", "valid", "val", "dev"]
    )
].dropna(subset=["text", "label"]).copy()

if valid_df.empty:
    raise ValueError(
        "Nie znaleziono danych walidacyjnych. "
        "Sprawdź wartości w kolumnie split."
    )
    
assert set(valid_df["label"]) <= set(LABELS), "Etykiety spoza LABELS - proszę poprawić LABEL_MAP."

N_PER_CLASS = 25   # 100 dokumentów; na samym CPU można zmniejszyć do 15 i to opisać

parts = [
    group.sample(n=min(N_PER_CLASS, len(group)), random_state=RANDOM_STATE)
    for _, group in valid_df.groupby("label", sort=True)
]
classification_sample = (
    pd.concat(parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
classification_sample.to_parquet(ARTIFACTS / "lab6_classification_sample.parquet", index=False)
display(classification_sample["label"].value_counts())

In [ ]:
def parse_label(text):
    """Akceptuje samą nazwę klasy, ewentualnie w cudzysłowie lub z kropką."""
    value = str(text).strip().lower().strip(' \t\r\n"\'`.')
    return value if value in LABELS else "INVALID"

assert parse_label("Nie jest pozytywna.") == "INVALID"
assert parse_label('"Pozytywna."') == "pozytywna"

def macro_f1_ci(y_true, y_pred, n_boot=1000):
    rng = np.random.default_rng(RANDOM_STATE)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        scores.append(f1_score(y_true[i], y_pred[i], labels=LABELS,
                               average="macro", zero_division=0))
    return np.percentile(scores, [2.5, 97.5])

In [ ]:
# Proszę zaimplementować wspólny benchmark klasyfikacji zero-shot.

# Zadanie 5: Wyjście strukturalne

Proszę wykorzystać przygotowany zestaw przypadków ekstrakcyjnych i porównać trzy tryby dla każdego modelu:

1. `free` -- sama instrukcja z opisem kluczy, bez parametru `format`;
2. `json` -- `format="json"`;
3. `schema` -- `format=IE_SCHEMA` (dekodowanie ograniczone schematem).

W trybie `schema` poprawność składni jest wymuszona przez dekoder, więc nie jest miarą jakości modelu. Proszę sprawdzić, czy wymuszenie formatu zmienia poprawność treści.

Dla każdego modelu proszę zmierzyć:

- poprawność składni JSON;
- zgodność ze schematem;
- poprawność pól;
- poprawność całego rekordu;
- średni czas odpowiedzi.

Proszę odróżnić błąd treści od błędu formatu.

In [ ]:

IE_CASES = [
    {"text": "W czwartek po południu padał deszcz.",
     "gold": {"person": None, "organization": None, "location": None}},
    {"text": "Uniwersytet Warszawski ogłosił nowy program stypendialny.",
     "gold": {"person": None, "organization": "Uniwersytet Warszawski", "location": None}},
    {"text": "Marii Skłodowskiej-Curie poświęcono wystawę w Paryżu.",
     "gold": {"person": "Marii Skłodowskiej-Curie", "organization": None, "location": "Paryżu"}},
    {"text": "Do Gdańska przyjechał Adam Małysz na zaproszenie Polskiego Związku Narciarskiego.",
     "gold": {"person": "Adam Małysz", "organization": "Polskiego Związku Narciarskiego", "location": "Gdańska"}},
    {"text": "Allegro otworzyło biuro w Poznaniu, a kieruje nim Ewa Nowak.",
     "gold": {"person": "Ewa Nowak", "organization": "Allegro", "location": "Poznaniu"}},
    {"text": "Prezes Orlenu spotkał się z dziennikarzami w Płocku.",
     "gold": {"person": None, "organization": "Orlenu", "location": "Płocku"}},
]

IE_SCHEMA = {
    "type": "object",
    "properties": {
        "person": {"anyOf": [{"type": "string"}, {"type": "null"}]},
        "organization": {"anyOf": [{"type": "string"}, {"type": "null"}]},
        "location": {"anyOf": [{"type": "string"}, {"type": "null"}]},
    },
    "required": ["person", "organization", "location"],
    "additionalProperties": False,
}

IE_PROMPT = '''Wyodrębnij z polskiego tekstu jedną osobę, jedną organizację i jedną lokalizację.
Zwracaj literalne fragmenty tekstu, w tej samej formie gramatycznej co w tekście. Jeśli kategorii nie ma, zwróć null.
Odpowiedz wyłącznie obiektem JSON z kluczami 'person', 'organization', 'location'.

Tekst:
'''


In [ ]:
def score_ie(raw, gold):
    out = {"json_ok": False, "schema_ok": False, "fields_ok": 0.0, "record_ok": False}
    try:
        obj = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return out                               # błąd formatu
    out["json_ok"] = True
    try:
        jsonschema.validate(obj, IE_SCHEMA)
        out["schema_ok"] = True
    except jsonschema.ValidationError:
        pass                            # błąd schematu
    if isinstance(obj, dict):
        out["fields_ok"] = sum((k in obj) and (obj[k] == v)
                               for k, v in gold.items()) / len(gold)
    out["record_ok"] = bool(out["schema_ok"] and out["fields_ok"] == 1.0)
    return out

assert score_ie("{}", {"person": None, "organization": None, "location": None})["fields_ok"] == 0.0

In [ ]:
# Proszę zaimplementować benchmark wyjścia strukturalnego.

# Zadanie 6: Dwie kwantyzacje tego samego modelu

Jeżeli lokalne środowisko na to pozwala, proszę wybrać jeden model dostępny w dwóch kwantyzacjach i porównać je na niewielkiej wspólnej próbce.

Należy porównać:

- rozmiar modelu;
- pamięć;
- TTFT;
- tokeny/s;
- macro-F1 w klasyfikacji;
- poprawność JSON.

Proszę sprawdzić, czy zaobserwowana różnica macro-F1 jest istotna praktycznie i jak duża jest niepewność oszacowania. Ponieważ oba modele oceniamy na tych samych dokumentach, proszę przeprowadzić również analizę sparowaną. Proszę dodatkowo podać porównanie sparowane na tych samych dokumentach:

- odsetek dokumentów, na których obie kwantyzacje dały tę samą predykcję;
- liczbę zmian „poprawna → błędna" i „błędna → poprawna".

Dostępne kwantyzacje danego modelu proszę sprawdzić na liście tagów w bibliotece Ollamy (np. wariant `q4_K_M` i `q8_0`).

Jeżeli druga kwantyzacja nie może zostać uruchomiona na dostępnym sprzęcie, proszę udokumentować ograniczenie i wykonać porównanie na podstawie dwóch konfiguracji różniących się rozmiarem modelu.

In [ ]:
# Proszę przeprowadzić kontrolowane porównanie dwóch kwantyzacji lub dwóch rozmiarów modelu.

# Zadanie 7: Porównanie z klasycznym baseline'em

Proszę zestawić najlepsze lokalne LLM-y z wynikiem `TF-IDF + Linear SVM` z laboratorium 2.

| system | macro-F1 | ms/dokument | pamięć / rozmiar | poprawność formatu |
|---|---:|---:|---:|---:|
| TF-IDF + Linear SVM | | | | nie dotyczy |
| LLM A | | | | |
| LLM B | | | | |

Baseline proszę ocenić na dokładnie tej samej próbce (`lab6_classification_sample.parquet`) i zmierzyć jego czas na tym samym sprzęcie. Jeżeli model z laboratorium 2 nie jest dostępny, proszę wytrenować go ponownie na zbiorze treningowym PolEmo 2.0.

Kolumnę „pamięć / rozmiar" proszę rozbić na dwie: rozmiar na dysku i pamięć po załadowaniu. Proszę zaznaczyć, że porównanie nie jest symetryczne: SVM widział dane treningowe, a LLM działa zero-shot.

In [ ]:
# Proszę przygotować wspólną tabelę jakości i kosztu.

# Zadanie 8: Analiza 20 przypadków

Proszę przygotować dokładnie 20 przypadków pokazujących ograniczenia lokalnych LLM-ów.

Co najmniej 10 powinno pochodzić z klasyfikacji, a co najmniej 5 z zadania strukturalnego (z dowolnego trybu: `free`, `json`, `schema`). Przypadki proszę wybierać z zapisanych surowych odpowiedzi. Jeżeli błędów strukturalnych jest mniej niż 5, proszę to odnotować i uzupełnić tabelę przypadkami z klasyfikacji.

Tabela:

| nr | model | zadanie | wejście | oczekiwany wynik | wynik modelu | typ problemu | komentarz |
|---:|---|---|---|---|---|---|---|

Proszę stosować krótkie kategorie, np. `WRONG_LABEL`, `FORMAT`, `SCHEMA`, `HALLUCINATION`, `OVERLONG`, `INCONSISTENT`, `OTHER`.


In [ ]:
# Proszę przygotować tabelę dokładnie 20 przeanalizowanych przypadków.

# Zadanie 9: Interpretacja wyników

To zadanie nie wymaga nowej implementacji.

Proszę odpowiedzieć:

1. Czy większy model był zawsze lepszy?
2. Jaki koszt był najbardziej odczuwalny: pamięć, TTFT czy szybkość generowania?
3. Czy kwantyzacja zmieniła jakość na tyle, by miało to znaczenie praktyczne?
4. Czy odpowiedzi poprawne formalnie były też poprawne merytorycznie? Proszę podać liczbę rekordów z poprawnym JSON i błędną treścią oraz porównać tryby `free`, `json` i `schema`.
5. Jakie dwa ograniczenia benchmarku najbardziej utrudniają uogólnienie wyników?

# Zadanie 10: Decyzja projektowa

To zadanie nie wymaga nowej implementacji.

Proszę wybrać model dla dwóch scenariuszy:

1. lokalny system, w którym najważniejsza jest jakość;
2. lokalny system, w którym ważne są mała pamięć i szybka odpowiedź.

Dla każdego wyboru proszę podać wynik jakościowy, koszt, najważniejszy typ błędu i jedno ograniczenie.

Na końcu proszę odpowiedzieć: **kiedy lokalny LLM jest uzasadnionym wyborem, a kiedy prostszy model zadaniowy pozostaje lepszym rozwiązaniem?**


# Karta modelu

Proszę utworzyć `artifacts/local_llm_model_card.md` dla modelu rekomendowanego do dalszych laboratoriów.

Karta powinna zawierać: nazwę i kwantyzację, środowisko, pamięć i rozmiar, TTFT i tokeny/s, wyniki klasyfikacji, wyniki wyjścia strukturalnego, porównanie z baseline'em, dominujące błędy z analizy 20 przypadków, zastosowanie, ograniczenia oraz uzasadnienie wyboru.


# Wnioski

Proszę zapisać KILKA krótkich wniosków opartych na uzyskanych liczbach i przykładach. Wnioski powinny obejmować jakość, koszt, kwantyzację, wyjście strukturalne oraz relację LLM–baseline.



# *Zadanie dla chętnych: polska fleksja i znaki diakrytyczne

Zadanie nie jest wymagane do zaliczenia. Sprawdza dwie rzeczy, których nie widać na przykład w benchmarkach anglojęzycznych.

## Część A: forma podstawowa encji

W zadaniu 5 model kopiował fragment tekstu. Teraz ma zwrócić encję w mianowniku: `Krakowie` → `Kraków`, `Polskiego Związku Narciarskiego` → `Polski Związek Narciarski`. Wymaga to wygenerowania poprawnej formy, a nie przepisania.

Proszę uruchomić `LEMMA_CASES` z promptem `LEMMA_PROMPT` w trybie `format=IE_SCHEMA` dla co najmniej dwóch modeli i policzyć:

- poprawność pól i całego rekordu (`score_ie` względem `gold_lemma`);
- rozkład typów błędów z `lemma_error_type` dla każdego modelu;
- poprawność osobno dla nazw jednowyrazowych i wielowyrazowych.

Należy uruchomić i przeprowadzić dwa osobne eksperymenty na tych samych ośmiu tekstach:
- Ekstrakcja literalna z IE_PROMPT, oceniana względem gold.
- Normalizacja do mianownika z LEMMA_PROMPT, oceniana względem gold_lemma.

Proszę odpowiedzieć:

1. O ile spada poprawność pól względem wariantu literalnego (ten sam przypadek oceniony względem `gold`)?
2. Który typ błędu dominuje: przepisanie formy odmienionej czy wygenerowanie formy błędnej?
3. Które zjawiska są najtrudniejsze: nazwy wielowyrazowe z uzgodnieniem, nazwy w liczbie mnogiej (`Kielc`), odmiana przymiotnikowa (`Zakopanego`), nazwa osoby wewnątrz nazwy organizacji?
4. Czy większy model lub wyższa kwantyzacja zmienia wynik bardziej niż w zadaniu 5?

## Część B: tekst bez znaków diakrytycznych

Użytkownicy często piszą bez polskich znaków. Proszę przepuścić `classification_sample` przez `strip_diacritics` i powtórzyć klasyfikację z zadania 4 dla jednego LLM-a oraz dla baseline'u `TF-IDF + Linear SVM` (bez ponownego trenowania).

Dla każdego systemu proszę podać:

- macro-F1 na tekście oryginalnym i bez diakrytyków;
- 95% przedział ufności sparowanej różnicy (`paired_macro_f1_diff_ci`);
- liczbę zmian „poprawna → błędna" i „błędna → poprawna";
- dla LLM-a: średnią `prompt_tokens` w obu wariantach.

Proszę odpowiedzieć:

1. Który system traci więcej i dlaczego? Proszę odnieść się do słownika TF-IDF i do tokenizacji podwyrazowej.
2. Czy usunięcie diakrytyków zmienia liczbę tokenów promptu, a więc koszt?
3. Proszę pokazać trzy dokumenty, w których predykcja się zmieniła, i wskazać słowa, które mogły o tym zdecydować (np. `łaska` / `laska`, `sąd` / `sad`).


In [ ]:
# Część A: gold = fragment literalny, gold_lemma = forma podstawowa (mianownik).
LEMMA_CASES = [
    {"text": "Marii Skłodowskiej-Curie poświęcono wystawę w Paryżu.",
     "gold": {"person": "Marii Skłodowskiej-Curie", "organization": None, "location": "Paryżu"},
     "gold_lemma": {"person": "Maria Skłodowska-Curie", "organization": None, "location": "Paryż"}},
    {"text": "Do Gdańska przyjechał Adam Małysz na zaproszenie Polskiego Związku Narciarskiego.",
     "gold": {"person": "Adam Małysz", "organization": "Polskiego Związku Narciarskiego", "location": "Gdańska"},
     "gold_lemma": {"person": "Adam Małysz", "organization": "Polski Związek Narciarski", "location": "Gdańsk"}},
    {"text": "Prezes Orlenu spotkał się z dziennikarzami w Płocku.",
     "gold": {"person": None, "organization": "Orlenu", "location": "Płocku"},
     "gold_lemma": {"person": None, "organization": "Orlen", "location": "Płock"}},
    {"text": "Nagrodę wręczono Januszowi Gajosowi w Teatrze Narodowym w Warszawie.",
     "gold": {"person": "Januszowi Gajosowi", "organization": "Teatrze Narodowym", "location": "Warszawie"},
     "gold_lemma": {"person": "Janusz Gajos", "organization": "Teatr Narodowy", "location": "Warszawa"}},
    {"text": "W Zielonej Górze spotkali się przedstawiciele Polskich Kolei Państwowych.",
     "gold": {"person": None, "organization": "Polskich Kolei Państwowych", "location": "Zielonej Górze"},
     "gold_lemma": {"person": None, "organization": "Polskie Koleje Państwowe", "location": "Zielona Góra"}},
    {"text": "Kibice z Kielc czekali na Roberta Lewandowskiego.",
     "gold": {"person": "Roberta Lewandowskiego", "organization": None, "location": "Kielc"},
     "gold_lemma": {"person": "Robert Lewandowski", "organization": None, "location": "Kielce"}},
    {"text": "Wycieczka do Zakopanego kosztowała Annę Kowalską majątek.",
     "gold": {"person": "Annę Kowalską", "organization": None, "location": "Zakopanego"},
     "gold_lemma": {"person": "Anna Kowalska", "organization": None, "location": "Zakopane"}},
    {"text": "Delegacja Uniwersytetu Marii Curie-Skłodowskiej odwiedziła Katowice.",
     "gold": {"person": None, "organization": "Uniwersytetu Marii Curie-Skłodowskiej", "location": "Katowice"},
     "gold_lemma": {"person": None, "organization": "Uniwersytet Marii Curie-Skłodowskiej", "location": "Katowice"}},
]

LEMMA_PROMPT = '''Wyodrębnij z polskiego tekstu jedną osobę, jedną organizację i jedną lokalizację.
Każdą nazwę podaj w formie podstawowej (mianownik), a nie w formie użytej w tekście. Jeśli kategorii nie ma, zwróć null.
Odpowiedz wyłącznie obiektem JSON z kluczami 'person', 'organization', 'location'.

Tekst:
'''


def lemma_error_type(pred, literal, lemma):
    """Typ wyniku dla jednego pola w części A."""
    if pred == lemma:
        return "OK"
    if lemma is None:
        return "HALLUCINATION"        # model podał encję, której nie ma
    if pred is None:
        return "MISSING"
    if pred == literal:
        return "COPIED_INFLECTED"     # przepisana forma odmieniona
    return "WRONG_FORM"               # inna, błędna forma lub inny fragment


# Część B: ł nie rozkłada się w Unicode, dlatego używamy jawnej tabeli.
_PL_DIACRITICS = str.maketrans("ąćęłńóśźżĄĆĘŁŃÓŚŹŻ", "acelnoszzACELNOSZZ")


def strip_diacritics(text):
    return str(text).translate(_PL_DIACRITICS)


assert strip_diacritics("Zażółć gęślą jaźń") == "Zazolc gesla jazn"
assert lemma_error_type("Krakowie", "Krakowie", "Kraków") == "COPIED_INFLECTED"
assert lemma_error_type("Kraków", "Krakowie", "Kraków") == "OK"

# Warunki zaliczenia laboratorium

- [ ] audyt środowiska i co najmniej dwóch modeli;
- [ ] oszacowanie i pomiar pamięci;
- [ ] cold start, TTFT i tokeny/s;
- [ ] benchmark klasyfikacji zero-shot;
- [ ] benchmark wyjścia strukturalnego w trzech trybach;
- [ ] surowe odpowiedzi zapisane w `artifacts/`;
- [ ] przedział ufności macro-F1 i porównanie sparowane kwantyzacji;
- [ ] porównanie dwóch kwantyzacji lub dwóch rozmiarów modelu;
- [ ] porównanie z `TF-IDF + Linear SVM`;
- [ ] tabela dokładnie 20 przypadków;
- [ ] odpowiedzi do dwóch zadań interpretacyjnych;
- [ ] `artifacts/local_llm_model_card.md`;
- [ ] wnioski.

Dla chętnych: zadanie o polskiej fleksji i znakach diakrytycznych (część A, część B lub obie).